# Step 22.2 — Learned positions → RoPE：用旋转表达相对位移

## 先建立独立工作副本（以下只是命令参考，本次未执行）

在仓库根目录先同步母版：

```bash
git pull
```

再创建不覆盖已有文件的时间戳副本（仓库应已有 notebooks/work 目录）：

```bash
cp -n notebooks/98_step22_2_rotary_position_embeddings.ipynb notebooks/work/98_step22_2_rotary_position_embeddings_$(date +%Y%m%d_%H%M%S)_work.ipynb
```

只在工作副本中填写练习或未来运行。`cp -n` 在同名目标存在时不覆盖；同一秒再次操作也不会覆盖已有内容。当前母版是**材料初稿 0/2**，不是学习完成证明。

**执行边界：**所有代码单元 execution_count 为 null、outputs 为空。本次不执行 notebook、MLX 实验、测试、训练、验证、采样、语料读取/下载或 checkpoint 操作。下面的短实验是将来可观察的安排，不是已观察结果。MLX API 基线固定 v0.32.2。

## 1. 为什么需要这一节

Learned absolute positions 给位置 p 一条可训练向量，再加到 token embedding 上，直观但受位置表容量约束。RoPE（rotary position embedding，旋转位置编码）把位置信息放进每个 attention head 的 Q/K：对相邻维度对施加位置相关的二维旋转。

它不是给 embedding 再加一张表，也通常不旋转 V。本节手写相邻配对的数学参考，便于看清公式，不替换原模型；未来若用 fused RoPE 必须核对 pairing convention。

## 2. 先备数学与 Python / MLX 语义

二维旋转 R(θ)=[[cosθ,−sinθ],[sinθ,cosθ]]，作用于 (a,b) 得 (a cosθ−b sinθ, a sinθ+b cosθ)。旋转保持长度与同角度旋转后的点积。

对 head dimension d 为偶数，维度对 j 使用频率 ωⱼ=base^(−2j/d)，位置 p 的角度为 pωⱼ。Qₚ'=R(pω)Qₚ，Kq'=R(qω)Kq，因此 Qₚ'ᵀKq'=QₚᵀR((q−p)ω)Kq。点积依赖相对位移 q−p。内容向量本身当然仍不同，不能说整个 attention 只取决于距离。

## 3. 纸笔走一遍（讲解例子，不是练习答案）

只看一个维度对，Q=[1,0]，K=[1,0]。若 Q 角度为 0，K 角度为 π/2，旋转后的点积为 0；若两者都额外旋转 π/3，点积仍为 0。这来自 cos(π/2)，不是运行结果。零位置角度为 0，旋转应保持原向量。

默认 base=10000 控制不同维度对的频率尺度。能计算任意位置的角度不代表模型已经学会任意长序列；训练长度、频率外推、数值精度与评测仍重要。

## 4. Shape tracing 与接口契约

Q/K (B,H,T,d) → reshape (B,H,T,d/2,2) → even/odd (B,H,T,d/2)。angles (T,d/2) 广播到 batch/head；stack 后恢复 (B,H,T,d)。position offset 是该 chunk 第一个 token 的绝对位置；增量 decode 时不能每轮都重置为 0。

In [ ]:
import math
import mlx.core as mx

# 本单元及后续所有单元均为未执行的教学源码。
# 仅在未来独立工作副本中按顺序运行；本次材料制作没有导入 MLX。

def rope_adjacent(x, *, offset=0, base=10000.0):
    if x.ndim != 4 or x.shape[-1] % 2 or x.shape[-1] == 0:
        raise ValueError("expect (B,H,T,d) with positive even d")
    if type(offset) is not int or offset < 0:
        raise ValueError("offset must be a nonnegative integer")
    if not math.isfinite(base) or base <= 0:
        raise ValueError("base must be positive and finite")
    B, H, T, d = x.shape
    inv_freq = mx.exp(-math.log(base) * mx.arange(0, d, 2, dtype=mx.float32) / d)
    positions = mx.arange(offset, offset + T, dtype=mx.float32)
    angles = positions[:, None] * inv_freq[None, :]
    c, s = mx.cos(angles), mx.sin(angles)
    pairs = x.astype(mx.float32).reshape(B, H, T, d // 2, 2)
    a, b = pairs[..., 0], pairs[..., 1]
    rotated = mx.stack([a * c - b * s, a * s + b * c], axis=-1)
    return rotated.reshape(B, H, T, d)

## 5. 短实验计划

先预测 shape 与趋势，再在未来工作副本中运行下一个单元。这里没有运行记录。若未来结果和推导不同，先检查输入、axis、dtype、mask、随机 key，再讨论数值误差；不要把一次随机输出当成分布证明。

In [ ]:
x = mx.array([[[[1., 0., 0., 1.], [1., 0., 0., 1.]]]])
y = rope_adjacent(x, offset=0)
before_norm2 = mx.sum(x * x, axis=-1)
after_norm2 = mx.sum(y * y, axis=-1)
mx.eval(y, before_norm2, after_norm2)
print("shape:", y.shape)
print("norm² before/after:", before_norm2.tolist(), after_norm2.tolist())

## 6. 常见误区与未来衔接

本函数用 float32 返回以突出数学，不承诺保持混合精度 dtype，也不支持部分维度 RoPE。生产实现还需明确 rotary dimensions、base、scale、offset、pairing、dtype。

MLX fast.rope 存在 traditional 配对参数；本节不调用它，该底层接口的 traditional 是必填 keyword，不能省略；避免将相邻配对与分半配对混为一谈。更换 position 机制意味着重新设计/训练或有依据的转换，不能将 learned position checkpoint 删除一张表后宣称等价。RoPE 与 KV cache 配合仍需保持位置编号和 mask 对齐。

## 7. 练习（不附答案）

1. 手算 (2,1) 旋转 π/2 后的坐标与平方范数。
2. 用 R 的转置与乘法解释相对位置公式。
3. 若已有 8 个 token 缓存，下一个单 token chunk 的 offset 应如何定义？说明依据。

请先用纸笔作答，再决定是否在工作副本中添加自己的代码。母版不填入参考答案。

## 8. 官方 API 阅读与完成边界

- [MLX v0.32.2 / python/src/fast.cpp](https://github.com/ml-explore/mlx/blob/v0.32.2/python/src/fast.cpp)
- [MLX v0.32.2 / python/src/ops.cpp](https://github.com/ml-explore/mlx/blob/v0.32.2/python/src/ops.cpp)

接口说明以固定版本源码为准。本次只有文本与源代码静态检查，没有运行通过声明。能够复述概念与完成练习后再进入下一节；材料覆盖范围不代表读者已经掌握。